# Document-Grounded Q&A System for Long Documents

**Objective:** Build an AI system that answers questions accurately from a single long document (100+ pages), with citation support and hallucination resistance.

**Architecture Overview:**
1. **Ingestion** — PDF parsing with structure extraction (headings, tables, page numbers)
2. **Chunking** — Heading-aware, overlapping chunks with metadata preservation
3. **Indexing** — Dual index: FAISS (dense vectors) + BM25 (sparse/keyword)
4. **Retrieval** — Hybrid search with Reciprocal Rank Fusion (RRF)
5. **Re-ranking** — Cross-encoder re-ranker for precision (optional)
6. **Generation** — LLM with strict grounding prompt + structured citations
7. **Evaluation** — Automated + manual evaluation harness

---
## Section 1: Setup & Configuration

### Dependencies
Install the required packages. We use:
- `pymupdf` (fitz) for PDF parsing — best balance of speed, structure extraction, and table support
- `sentence-transformers` for dense embeddings — local, no API dependency
- `faiss-cpu` for vector similarity search — efficient ANN for single-doc scale
- `rank_bm25` for sparse keyword retrieval — complements semantic search
- `openai` for LLM generation — easily swappable for other providers
- `cross-encoder` model via sentence-transformers for re-ranking

In [ ]:
# Install dependencies
!pip install pymupdf sentence-transformers faiss-cpu rank_bm25 openai tiktoken tabulate dotenv -q

In [15]:
import os
import re
import json
import hashlib
import textwrap
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Tuple, Any
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

# PDF parsing
import fitz  # PyMuPDF

# Embeddings & vector search
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Sparse retrieval
from rank_bm25 import BM25Okapi

# LLM
from openai import OpenAI

# Tokenizer for chunk sizing
import tiktoken

load_dotenv()  # Load environment variables from .env file

print("All imports successful.")

All imports successful.


In [9]:
# ── Configuration ──────────────────────────────────────────────
# Centralizing all hyperparameters for easy tuning and documentation.

@dataclass
class Config:
    # --- Paths ---
    pdf_path: str = "document.pdf"             # Path to the input PDF
    index_dir: str = "./index_cache"           # Cache directory for indices

    # --- Chunking ---
    chunk_size: int = 512                      # Target chunk size in tokens
    chunk_overlap: int = 64                    # Overlap between consecutive chunks (tokens)
    min_chunk_size: int = 50                   # Discard chunks smaller than this

    # --- Embedding ---
    embedding_model: str = "BAAI/bge-base-en-v1.5"  # Dense embedding model
    # Why bge-base: strong retrieval performance, 768-dim, good balance of
    # quality vs speed. Alternatives: e5-large-v2 (better but slower),
    # all-MiniLM-L6-v2 (faster but weaker on long passages).

    # --- Retrieval ---
    top_k_initial: int = 20                    # Candidates from each retriever
    top_k_rerank: int = 8                      # Final passages after re-ranking
    rrf_k: int = 60                            # RRF smoothing constant
    bm25_weight: float = 1.0                   # Weight for BM25 in fusion
    dense_weight: float = 1.0                  # Weight for dense retrieval in fusion

    # --- Re-ranking ---
    use_reranker: bool = True
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    # Why this reranker: lightweight cross-encoder, good accuracy/speed tradeoff.
    # For production: consider ms-marco-MiniLM-L-12-v2 or a larger model.

    # --- Generation ---
    llm_model: str = "gpt-4o-mini"             # LLM for answer generation
    # gpt-4o-mini: cost-effective, strong instruction following.
    # Swap to gpt-4o or claude-3.5-sonnet for higher quality.
    temperature: float = 0.0                   # Deterministic for factual QA
    max_answer_tokens: int = 1024

    # --- OpenAI ---
    openai_api_key: str = ""                   # Set via env var OPENAI_API_KEY

config = Config()

# Load API key from environment
config.openai_api_key = os.environ.get("OPENAI_API_KEY", "")
if not config.openai_api_key:
    print("⚠️  OPENAI_API_KEY not set. Set it before running generation cells.")
    print("   os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print("✅ OpenAI API key loaded.")

print(f"\nConfig summary:")
print(f"  Embedding model : {config.embedding_model}")
print(f"  Chunk size      : {config.chunk_size} tokens (overlap {config.chunk_overlap})")
print(f"  Retrieval top-k : {config.top_k_initial} → rerank to {config.top_k_rerank}")
print(f"  Re-ranker       : {'ON' if config.use_reranker else 'OFF'}")
print(f"  LLM             : {config.llm_model}")

✅ OpenAI API key loaded.

Config summary:
  Embedding model : BAAI/bge-base-en-v1.5
  Chunk size      : 512 tokens (overlap 64)
  Retrieval top-k : 20 → rerank to 8
  Re-ranker       : ON
  LLM             : gpt-4o-mini


---
## Section 2: Document Ingestion

### Design Decisions
- **PyMuPDF** over alternatives (pdfplumber, pdfminer, unstructured):
  - Fast C-based parser, handles most PDF layouts well
  - Built-in table extraction (since v1.23)
  - Preserves reading order and bounding boxes
  - Gives us page-level metadata for citations
- We extract: raw text per page, headings (via font-size heuristics), and tables
- Each "block" of content gets a `PageElement` with source metadata attached

In [12]:
@dataclass
class PageElement:
    """A single element extracted from a PDF page."""
    text: str
    page_num: int              # 1-indexed page number
    element_type: str          # 'heading', 'paragraph', 'table', 'list_item'
    heading_level: int = 0     # 1-3 for headings, 0 for non-headings
    section_title: str = ""    # The nearest heading above this element
    bbox: Optional[Tuple[float, float, float, float]] = None  # (x0, y0, x1, y1)

    def __repr__(self):
        return f"PageElement(p{self.page_num}, {self.element_type}, {len(self.text)} chars)"


class DocumentIngester:
    """Extract structured content from a PDF document."""

    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path
        self.doc = fitz.open(pdf_path)
        self.total_pages = len(self.doc)
        self.elements: List[PageElement] = []
        self.metadata: Dict[str, Any] = {}

    def _detect_heading(self, block_dict: dict, page_fonts: dict) -> Tuple[bool, int]:
        """
        Heuristic heading detection based on font size relative to page median.

        We compute the median font size per page, then classify lines with
        font sizes significantly above the median as headings.
        Level 1: >= 1.6x median, Level 2: >= 1.3x median, Level 3: >= 1.15x median.
        """
        if "lines" not in block_dict:
            return False, 0

        # Get the dominant font size of this block
        font_sizes = []
        for line in block_dict["lines"]:
            for span in line["spans"]:
                font_sizes.append(span["size"])

        if not font_sizes:
            return False, 0

        block_font_size = max(font_sizes)  # Use max to catch heading spans
        median_size = page_fonts.get("median", 12)

        # Check if text is short enough to be a heading (not a paragraph in large font)
        full_text = " ".join(
            span["text"]
            for line in block_dict["lines"]
            for span in line["spans"]
        ).strip()

        if len(full_text) > 200:  # Too long to be a heading
            return False, 0

        # Check for bold as additional signal
        is_bold = any(
            "bold" in span.get("font", "").lower()
            for line in block_dict["lines"]
            for span in line["spans"]
        )

        ratio = block_font_size / median_size if median_size > 0 else 1.0

        if ratio >= 1.6 or (ratio >= 1.4 and is_bold):
            return True, 1
        elif ratio >= 1.3 or (ratio >= 1.15 and is_bold):
            return True, 2
        elif ratio >= 1.15 or (is_bold and len(full_text) < 80):
            return True, 3

        return False, 0

    def _get_page_font_stats(self, page) -> dict:
        """Compute median font size for a page (used for heading detection)."""
        sizes = []
        blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]
        for b in blocks:
            if b["type"] == 0:  # text block
                for line in b["lines"]:
                    for span in line["spans"]:
                        text = span["text"].strip()
                        if len(text) > 2:  # Skip tiny fragments
                            sizes.extend([span["size"]] * len(text))
        if sizes:
            return {"median": float(np.median(sizes))}
        return {"median": 12.0}

    def _extract_tables(self, page, page_num: int) -> List[PageElement]:
        """Extract tables from a page using PyMuPDF's table finder."""
        elements = []
        try:
            tables = page.find_tables()
            for table in tables:
                # Convert table to markdown format for LLM consumption
                df = table.to_pandas()
                if df.empty:
                    continue
                # Format as markdown table
                md_table = df.to_markdown(index=False)
                elements.append(PageElement(
                    text=md_table,
                    page_num=page_num,
                    element_type="table",
                    bbox=tuple(table.bbox) if table.bbox else None
                ))
        except Exception:
            pass  # Table extraction can fail on some PDFs; degrade gracefully
        return elements

    def ingest(self) -> List[PageElement]:
        """
        Extract all elements from the PDF.

        Returns a list of PageElement objects in reading order,
        with headings, paragraphs, and tables identified.
        """
        self.elements = []
        current_section = ""

        # Document-level metadata
        meta = self.doc.metadata
        self.metadata = {
            "title": meta.get("title", ""),
            "author": meta.get("author", ""),
            "total_pages": self.total_pages,
            "file_name": os.path.basename(self.pdf_path),
        }

        for page_idx in range(self.total_pages):
            page = self.doc[page_idx]
            page_num = page_idx + 1
            font_stats = self._get_page_font_stats(page)

            # Extract text blocks
            blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]

            # Extract tables (we'll use their bboxes to avoid duplicating table text)
            table_elements = self._extract_tables(page, page_num)
            table_bboxes = [te.bbox for te in table_elements if te.bbox]

            for block in blocks:
                if block["type"] != 0:  # Skip image blocks
                    continue

                block_bbox = tuple(block.get("bbox", (0, 0, 0, 0)))

                # Skip if this block overlaps with an extracted table
                if self._bbox_overlaps_any(block_bbox, table_bboxes):
                    continue

                # Reconstruct text from spans
                full_text = ""
                for line in block["lines"]:
                    line_text = "".join(span["text"] for span in line["spans"])
                    full_text += line_text + "\n"
                full_text = full_text.strip()

                if not full_text or len(full_text) < 3:
                    continue

                # Detect headings
                is_heading, level = self._detect_heading(block, font_stats)

                if is_heading:
                    current_section = full_text.strip()
                    self.elements.append(PageElement(
                        text=full_text,
                        page_num=page_num,
                        element_type="heading",
                        heading_level=level,
                        section_title=current_section,
                        bbox=block_bbox
                    ))
                else:
                    self.elements.append(PageElement(
                        text=full_text,
                        page_num=page_num,
                        element_type="paragraph",
                        section_title=current_section,
                        bbox=block_bbox
                    ))

            # Add table elements
            for te in table_elements:
                te.section_title = current_section
                self.elements.append(te)

        print(f"✅ Ingested {self.total_pages} pages → {len(self.elements)} elements")
        element_types = {}
        for e in self.elements:
            element_types[e.element_type] = element_types.get(e.element_type, 0) + 1
        for t, c in sorted(element_types.items()):
            print(f"   {t}: {c}")

        return self.elements

    @staticmethod
    def _bbox_overlaps_any(bbox, others, threshold=0.5) -> bool:
        """Check if bbox significantly overlaps with any bbox in others."""
        if not others:
            return False
        x0, y0, x1, y1 = bbox
        area = max((x1 - x0) * (y1 - y0), 1e-6)
        for ox0, oy0, ox1, oy1 in others:
            ix0 = max(x0, ox0)
            iy0 = max(y0, oy0)
            ix1 = min(x1, ox1)
            iy1 = min(y1, oy1)
            if ix0 < ix1 and iy0 < iy1:
                intersection = (ix1 - ix0) * (iy1 - iy0)
                if intersection / area > threshold:
                    return True
        return False

    def close(self):
        self.doc.close()

print("DocumentIngester defined.")

DocumentIngester defined.


In [16]:
# ── Run Ingestion ──────────────────────────────────────────────
# Point this to your PDF file. For testing, download a 10-K filing:
# e.g., https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm
# or use any 100+ page PDF you have.

PDF_PATH = "document.pdf"  # ← CHANGE THIS to your PDF path

# Verify the file exists
if not os.path.exists(PDF_PATH):
    print(f"❌ File not found: {PDF_PATH}")
    print("   Please set PDF_PATH to a valid PDF file path.")
    print("   Example: PDF_PATH = '/path/to/your/annual_report.pdf'")
else:
    config.pdf_path = PDF_PATH
    ingester = DocumentIngester(PDF_PATH)
    elements = ingester.ingest()

    # Quick preview
    print(f"\n📄 Document: {ingester.metadata.get('title', 'N/A')}")
    print(f"   Pages: {ingester.total_pages}")
    print(f"\n--- Sample Elements ---")
    for e in elements[:5]:
        preview = e.text[:100].replace('\n', ' ')
        print(f"  [p{e.page_num}] ({e.element_type}) {preview}...")

✅ Ingested 160 pages → 816 elements
   heading: 185
   paragraph: 626
   table: 5

📄 Document: 10-K - 02/07/2019 - 3M Company
   Pages: 160

--- Sample Elements ---
  [p1] (paragraph) low...
  [p1] (heading) UNITED STATES SECURITIES AND EXCHANGE COMMISSION...
  [p1] (heading) Washington, D.C. 20549...
  [p1] (heading) FORM 10-K...
  [p1] (heading) ☒   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE...


---
## Section 3: Structure-Aware Chunking

### Strategy
The chunking approach is critical for retrieval quality. Key decisions:

1. **Heading-first splitting**: We first group elements by their section headings.
   This preserves topical coherence — a chunk about "Revenue Recognition" won't
   bleed into "Risk Factors."

2. **Token-based sizing**: We use tiktoken to count tokens (matching the LLM's tokenizer)
   rather than characters. This ensures chunks fit well in context windows.

3. **Overlap**: Adjacent chunks share `chunk_overlap` tokens to avoid losing information
   at chunk boundaries. This is especially important for sentences that span boundaries.

4. **Metadata preservation**: Every chunk carries its source page numbers, section title,
   and element types. This is essential for citation generation later.

### Alternatives Considered
- **Sentence-level splitting**: More granular but loses paragraph context.
- **Recursive character splitting** (LangChain-style): Simpler but heading-unaware.
- **Semantic chunking** (embed sentences, split at semantic boundaries): Higher quality
  but much slower and adds complexity. Worth exploring in v2.

In [17]:
@dataclass
class Chunk:
    """A chunk of document content ready for indexing."""
    chunk_id: str                      # Unique identifier
    text: str                          # The chunk content
    page_numbers: List[int]            # Source page(s)
    section_title: str                 # Nearest section heading
    element_types: List[str]           # Types of elements in this chunk
    token_count: int                   # Number of tokens
    chunk_index: int                   # Position in document order

    def citation_str(self) -> str:
        """Format citation for this chunk."""
        pages = sorted(set(self.page_numbers))
        if len(pages) == 1:
            page_str = f"p. {pages[0]}"
        else:
            page_str = f"pp. {pages[0]}-{pages[-1]}"
        section = f' — "{self.section_title}"' if self.section_title else ""
        return f"[{page_str}{section}]"


class StructureAwareChunker:
    """Split document elements into overlapping, heading-aware chunks."""

    def __init__(self, config: Config):
        self.config = config
        self.tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

    def _count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text))

    def _split_text_by_tokens(self, text: str, max_tokens: int) -> List[str]:
        """Split a long text into pieces that fit within max_tokens."""
        tokens = self.tokenizer.encode(text)
        pieces = []
        for i in range(0, len(tokens), max_tokens):
            piece_tokens = tokens[i:i + max_tokens]
            pieces.append(self.tokenizer.decode(piece_tokens))
        return pieces

    def chunk(self, elements: List[PageElement]) -> List[Chunk]:
        """
        Convert elements into chunks using a two-phase approach:
        Phase 1: Group elements by section heading.
        Phase 2: Split groups into token-bounded chunks with overlap.
        """
        # Phase 1: Group by section
        sections = self._group_by_section(elements)

        # Phase 2: Create chunks from each section
        chunks = []
        chunk_idx = 0

        for section_title, section_elements in sections:
            section_chunks = self._chunk_section(
                section_title, section_elements, chunk_idx
            )
            chunks.extend(section_chunks)
            chunk_idx += len(section_chunks)

        # Filter out tiny chunks
        chunks = [c for c in chunks if c.token_count >= self.config.min_chunk_size]

        print(f"✅ Created {len(chunks)} chunks")
        token_counts = [c.token_count for c in chunks]
        print(f"   Token stats: min={min(token_counts)}, max={max(token_counts)}, "
              f"mean={np.mean(token_counts):.0f}, median={np.median(token_counts):.0f}")

        return chunks

    def _group_by_section(self, elements: List[PageElement]) -> List[Tuple[str, List[PageElement]]]:
        """Group elements under their nearest heading."""
        sections = []
        current_title = ""
        current_elements = []

        for elem in elements:
            if elem.element_type == "heading" and elem.heading_level <= 2:
                # Start a new section
                if current_elements:
                    sections.append((current_title, current_elements))
                current_title = elem.text.strip()
                current_elements = [elem]
            else:
                current_elements.append(elem)

        if current_elements:
            sections.append((current_title, current_elements))

        return sections

    def _chunk_section(
        self, section_title: str, elements: List[PageElement], start_idx: int
    ) -> List[Chunk]:
        """Split a section's elements into token-bounded chunks with overlap."""
        chunks = []
        current_texts = []
        current_pages = []
        current_types = []
        current_tokens = 0

        for elem in elements:
            elem_tokens = self._count_tokens(elem.text)

            # If a single element exceeds chunk_size, split it
            if elem_tokens > self.config.chunk_size:
                # Flush current buffer first
                if current_texts:
                    chunks.append(self._make_chunk(
                        current_texts, current_pages, current_types,
                        section_title, start_idx + len(chunks)
                    ))
                    current_texts, current_pages, current_types = [], [], []
                    current_tokens = 0

                # Split the large element
                pieces = self._split_text_by_tokens(elem.text, self.config.chunk_size)
                for piece in pieces:
                    chunks.append(self._make_chunk(
                        [piece], [elem.page_num], [elem.element_type],
                        section_title, start_idx + len(chunks)
                    ))
                continue

            # Check if adding this element would exceed the chunk size
            if current_tokens + elem_tokens > self.config.chunk_size and current_texts:
                chunks.append(self._make_chunk(
                    current_texts, current_pages, current_types,
                    section_title, start_idx + len(chunks)
                ))

                # Implement overlap: keep some trailing text
                overlap_texts, overlap_pages, overlap_types, overlap_tokens = \
                    self._compute_overlap(current_texts, current_pages, current_types)
                current_texts = overlap_texts
                current_pages = overlap_pages
                current_types = overlap_types
                current_tokens = overlap_tokens

            current_texts.append(elem.text)
            current_pages.append(elem.page_num)
            current_types.append(elem.element_type)
            current_tokens += elem_tokens

        # Flush remaining
        if current_texts:
            chunks.append(self._make_chunk(
                current_texts, current_pages, current_types,
                section_title, start_idx + len(chunks)
            ))

        return chunks

    def _compute_overlap(
        self, texts: List[str], pages: List[int], types: List[str]
    ) -> Tuple[List[str], List[int], List[str], int]:
        """Get the trailing elements that fit within the overlap budget."""
        overlap_budget = self.config.chunk_overlap
        result_texts, result_pages, result_types = [], [], []
        total_tokens = 0

        for text, page, etype in zip(reversed(texts), reversed(pages), reversed(types)):
            t = self._count_tokens(text)
            if total_tokens + t > overlap_budget:
                break
            result_texts.insert(0, text)
            result_pages.insert(0, page)
            result_types.insert(0, etype)
            total_tokens += t

        return result_texts, result_pages, result_types, total_tokens

    def _make_chunk(
        self, texts: List[str], pages: List[int], types: List[str],
        section_title: str, chunk_index: int
    ) -> Chunk:
        combined = "\n\n".join(texts)
        chunk_id = hashlib.md5(f"{chunk_index}:{combined[:100]}".encode()).hexdigest()[:12]
        return Chunk(
            chunk_id=chunk_id,
            text=combined,
            page_numbers=pages,
            section_title=section_title,
            element_types=list(set(types)),
            token_count=self._count_tokens(combined),
            chunk_index=chunk_index,
        )

print("StructureAwareChunker defined.")

StructureAwareChunker defined.


In [18]:
# ── Run Chunking ──────────────────────────────────────────────
if 'elements' in dir():
    chunker = StructureAwareChunker(config)
    chunks = chunker.chunk(elements)

    # Preview chunks
    print("\n--- Sample Chunks ---")
    for c in chunks[:3]:
        print(f"\n[Chunk {c.chunk_index}] {c.citation_str()} ({c.token_count} tokens)")
        print(f"  Section: {c.section_title[:60]}")
        print(f"  Preview: {c.text[:150].replace(chr(10), ' ')}...")
else:
    print("⚠️  No elements found. Run the ingestion cell first.")

✅ Created 348 chunks
   Token stats: min=52, max=538, mean=398, median=476

--- Sample Chunks ---

[Chunk 2] [p. 1 — "FORM 10-K"] (59 tokens)
  Section: FORM 10-K
  Preview: FORM 10-K  ☒   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE  SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2018  Comm...

[Chunk 3] [p. 1 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"] (483 tokens)
  Section: 3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employe
  Preview: 3M COMPANY State of Incorporation: Delaware   I.R.S. Employer Identification No. 41-0417775 Principal executive offices: 3M Center, St. Paul, Minnesot...

[Chunk 4] [p. 1 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"] (451 tokens)
  Section: 3M COMPANY
State of Incorporation: Del

---
## Section 4: Dual Index Construction

### Why Hybrid Retrieval?
Dense (embedding) and sparse (BM25) retrieval have complementary strengths:

| Aspect | Dense (FAISS) | Sparse (BM25) |
|--------|--------------|---------------|
| Strength | Semantic similarity, paraphrase handling | Exact keyword matching, rare terms |
| Weakness | Can miss exact terms, entity names | No semantic understanding |
| Example | "company revenue" ↔ "total sales" | "GAAP" must match "GAAP" exactly |

For financial/technical documents, exact term matching is critical (ticker symbols,
accounting terms, specific numbers), so BM25 is not just a nice-to-have — it's essential.

### Embedding Model Choice
We use `BAAI/bge-base-en-v1.5`:
- MTEB benchmark leader in its size class
- 768 dimensions — good quality without excessive memory
- Supports instruction-prefixed queries for better retrieval

In [19]:
class DualIndex:
    """
    Builds and queries a hybrid index over document chunks.
    Combines FAISS (dense vectors) with BM25 (sparse keyword matching).
    """

    def __init__(self, config: Config):
        self.config = config
        self.chunks: List[Chunk] = []

        # Dense index components
        self.embedding_model = None
        self.faiss_index = None
        self.embeddings = None

        # Sparse index components
        self.bm25 = None
        self.tokenized_corpus = None

    def build(self, chunks: List[Chunk]):
        """Build both dense and sparse indices from chunks."""
        self.chunks = chunks
        texts = [c.text for c in chunks]

        print("Building dense index...")
        self._build_dense_index(texts)

        print("Building sparse index...")
        self._build_sparse_index(texts)

        print(f"✅ Dual index built: {len(chunks)} chunks indexed")

    def _build_dense_index(self, texts: List[str]):
        """Embed all chunks and build a FAISS index."""
        self.embedding_model = SentenceTransformer(self.config.embedding_model)

        # bge models benefit from instruction prefix for passages
        # (queries get a different prefix at search time)
        self.embeddings = self.embedding_model.encode(
            texts,
            show_progress_bar=True,
            batch_size=32,
            normalize_embeddings=True  # For cosine similarity via inner product
        )

        # Build FAISS index with inner product (equivalent to cosine for normalized vectors)
        dim = self.embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dim)
        self.faiss_index.add(self.embeddings.astype(np.float32))

        print(f"   FAISS index: {self.faiss_index.ntotal} vectors, {dim} dimensions")

    def _build_sparse_index(self, texts: List[str]):
        """Build BM25 index from tokenized chunk texts."""
        # Simple whitespace + lowercasing tokenization
        # For production: consider stemming, stopword removal
        self.tokenized_corpus = [self._tokenize(t) for t in texts]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        print(f"   BM25 index: {len(self.tokenized_corpus)} documents")

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        """Simple tokenization for BM25. Production would use a proper tokenizer."""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        tokens = text.split()
        return [t for t in tokens if len(t) > 1]  # Remove single chars

    def search_dense(self, query: str, top_k: int) -> List[Tuple[int, float]]:
        """Search the FAISS index. Returns [(chunk_index, score), ...]."""
        # bge models: prefix query with instruction for better retrieval
        query_prefix = "Represent this sentence for searching relevant passages: "
        q_embedding = self.embedding_model.encode(
            [query_prefix + query],
            normalize_embeddings=True
        ).astype(np.float32)

        scores, indices = self.faiss_index.search(q_embedding, top_k)
        results = [(int(idx), float(score))
                   for idx, score in zip(indices[0], scores[0])
                   if idx >= 0]
        return results

    def search_sparse(self, query: str, top_k: int) -> List[Tuple[int, float]]:
        """Search the BM25 index. Returns [(chunk_index, score), ...]."""
        query_tokens = self._tokenize(query)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = [(int(idx), float(scores[idx]))
                   for idx in top_indices if scores[idx] > 0]
        return results

    def search_hybrid(self, query: str, top_k: int = None) -> List[Tuple[Chunk, float]]:
        """
        Hybrid search using Reciprocal Rank Fusion (RRF).

        RRF combines rankings from multiple retrievers without needing
        score normalization. Formula: RRF(d) = Σ 1/(k + rank_i(d))
        where k is a smoothing constant (typically 60).
        """
        if top_k is None:
            top_k = self.config.top_k_initial

        dense_results = self.search_dense(query, top_k)
        sparse_results = self.search_sparse(query, top_k)

        # RRF fusion
        rrf_scores = {}
        k = self.config.rrf_k

        for rank, (idx, _) in enumerate(dense_results):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + \
                self.config.dense_weight * (1.0 / (k + rank + 1))

        for rank, (idx, _) in enumerate(sparse_results):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + \
                self.config.bm25_weight * (1.0 / (k + rank + 1))

        # Sort by RRF score
        sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        return [(self.chunks[idx], score) for idx, score in sorted_results[:top_k]]

print("DualIndex defined.")

DualIndex defined.


In [20]:
# ── Build Index ──────────────────────────────────────────────
if 'chunks' in dir():
    dual_index = DualIndex(config)
    dual_index.build(chunks)
else:
    print("⚠️  No chunks found. Run the chunking cell first.")

Building dense index...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16749.61it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 11/11 [00:04<00:00,  2.74it/s]

   FAISS index: 348 vectors, 768 dimensions
Building sparse index...
   BM25 index: 348 documents
✅ Dual index built: 348 chunks indexed


In [22]:
# ── Test Retrieval ────────────────────────────────────────────
# Quick sanity check on retrieval quality before moving to generation.

if 'dual_index' in dir():
    test_query = "What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement?"  # ← Adjust for your document

    print(f"Query: '{test_query}'\n")

    # Compare dense vs sparse vs hybrid
    print("=== Dense (semantic) ===")
    for idx, score in dual_index.search_dense(test_query, 3):
        c = dual_index.chunks[idx]
        print(f"  [{score:.3f}] {c.citation_str()} {c.text[:100].replace(chr(10), ' ')}...")

    print("\n=== Sparse (BM25) ===")
    for idx, score in dual_index.search_sparse(test_query, 3):
        c = dual_index.chunks[idx]
        print(f"  [{score:.3f}] {c.citation_str()} {c.text[:100].replace(chr(10), ' ')}...")

    print("\n=== Hybrid (RRF) ===")
    for chunk, score in dual_index.search_hybrid(test_query, 3):
        print(f"  [{score:.4f}] {chunk.citation_str()} {chunk.text[:100].replace(chr(10), ' ')}...")

Query: 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement?'

=== Dense (semantic) ===
  [0.710] [p. 44 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"] Table of Contents  Cash, Cash Equivalents and Marketable Securities:   At December 31, 2018, 3M had ...
  [0.706] [p. 49 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"] . Refer to the preceding “Cash Flows from Investing Activities” section for discussion on capital sp...
  [0.690] [p. 47 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"] Investments in property, plant and equipment enable g

---
## Section 5: Re-Ranking (Optional but Recommended)

### Rationale
First-stage retrieval (FAISS + BM25) optimizes for **recall** — cast a wide net.
Re-ranking optimizes for **precision** — pick the best passages from that net.

A **cross-encoder** re-ranker jointly encodes (query, passage) pairs and produces
a relevance score. This is much more accurate than bi-encoder similarity but too
slow to run over the full corpus — hence the two-stage design.

### Impact
In practice, re-ranking consistently improves answer quality by:
- Pushing truly relevant passages to the top
- Demoting passages that share vocabulary but aren't actually relevant
- Handling complex queries where semantic similarity alone is insufficient

In [23]:
class Reranker:
    """Cross-encoder re-ranker for improving retrieval precision."""

    def __init__(self, config: Config):
        self.config = config
        self.model = None
        if config.use_reranker:
            print(f"Loading re-ranker: {config.reranker_model}")
            self.model = CrossEncoder(config.reranker_model, max_length=512)
            print("✅ Re-ranker loaded.")

    def rerank(
        self, query: str, candidates: List[Tuple[Chunk, float]], top_k: int = None
    ) -> List[Tuple[Chunk, float]]:
        """
        Re-rank candidate chunks using the cross-encoder.

        Args:
            query: The user question
            candidates: List of (Chunk, initial_score) from first-stage retrieval
            top_k: Number of results to return after re-ranking

        Returns:
            Re-ranked list of (Chunk, reranker_score)
        """
        if not self.config.use_reranker or self.model is None:
            return candidates[:top_k] if top_k else candidates

        if top_k is None:
            top_k = self.config.top_k_rerank

        # Prepare (query, passage) pairs for the cross-encoder
        pairs = [(query, chunk.text) for chunk, _ in candidates]

        # Score all pairs
        scores = self.model.predict(pairs, show_progress_bar=False)

        # Sort by cross-encoder score
        scored = list(zip(candidates, scores))
        scored.sort(key=lambda x: x[1], reverse=True)

        reranked = [(chunk, float(ce_score))
                    for (chunk, _), ce_score in scored[:top_k]]

        return reranked

print("Reranker defined.")

Reranker defined.


In [24]:
# ── Initialize Re-ranker ──────────────────────────────────────
if 'config' in dir():
    reranker = Reranker(config)

Loading re-ranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11492.74it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Re-ranker loaded.


---
## Section 6: Answer Generation with Citations

### Prompt Design for Grounding
The generation prompt is designed to minimize hallucination:

1. **Explicit instruction** to only use provided context
2. **Citation format** specified clearly (page numbers, section titles)
3. **Refusal instruction** — if evidence is insufficient, say so
4. **No inference beyond document** — prevents the LLM from using its own knowledge

### Citation Strategy
We attach metadata to each retrieved passage and instruct the LLM to cite
using `[Source N, p. X]` format. Post-processing extracts and validates citations.

### Why gpt-4o-mini?
- Strong instruction following (crucial for grounding)
- Cost-effective for iterative development
- Easily swappable: change `config.llm_model` to `gpt-4o` or use Anthropic's API

In [25]:
class AnswerGenerator:
    """Generate grounded answers with citations from retrieved passages."""

    SYSTEM_PROMPT = """You are a precise document analysis assistant. Your task is to answer
questions based ONLY on the provided source passages from a document.

RULES:
1. Use ONLY information from the provided source passages. Do not use any external knowledge.
2. Cite your sources using [Source N] notation, where N is the source number.
3. If multiple sources support a claim, cite all of them: [Source 1, Source 3].
4. If the provided passages do not contain enough information to answer the question,
   explicitly state: "The provided document sections do not contain sufficient information
   to answer this question."
5. Do not speculate or infer beyond what is explicitly stated in the sources.
6. For numerical data, quote the exact figures from the sources.
7. Keep your answer concise but complete."""

    def __init__(self, config: Config):
        self.config = config
        self.client = None
        if config.openai_api_key:
            self.client = OpenAI(api_key=config.openai_api_key)

    def _format_context(self, passages: List[Tuple[Chunk, float]]) -> str:
        """Format retrieved passages into a numbered context block."""
        context_parts = []
        for i, (chunk, score) in enumerate(passages, 1):
            citation = chunk.citation_str()
            context_parts.append(
                f"[Source {i}] {citation}\n"
                f"{chunk.text}\n"
            )
        return "\n---\n".join(context_parts)

    def generate(
        self, query: str, passages: List[Tuple[Chunk, float]]
    ) -> Dict[str, Any]:
        """
        Generate a grounded answer with citations.

        Returns:
            {
                "answer": str,           # The generated answer
                "sources": [             # Source passages used
                    {"source_num": int, "chunk": Chunk, "score": float}, ...
                ],
                "query": str,
                "model": str,
            }
        """
        if not self.client:
            return {
                "answer": "❌ OpenAI API key not configured. Please set OPENAI_API_KEY.",
                "sources": [],
                "query": query,
                "model": "none",
            }

        context = self._format_context(passages)

        user_message = f"""Based on the following source passages from the document, answer the question.

SOURCE PASSAGES:
{context}

QUESTION: {query}

Provide a precise, well-cited answer using only the information in the sources above."""

        response = self.client.chat.completions.create(
            model=self.config.llm_model,
            temperature=self.config.temperature,
            max_tokens=self.config.max_answer_tokens,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
        )

        answer_text = response.choices[0].message.content

        # Build source list
        sources = [
            {"source_num": i + 1, "chunk": chunk, "score": score}
            for i, (chunk, score) in enumerate(passages)
        ]

        return {
            "answer": answer_text,
            "sources": sources,
            "query": query,
            "model": self.config.llm_model,
            "usage": {
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
            }
        }

print("AnswerGenerator defined.")

AnswerGenerator defined.


---
## Section 7: Complete Q&A Pipeline

Ties all components together into a single callable pipeline.
This is the main interface for the system.

In [26]:
class DocumentQAPipeline:
    """End-to-end document Q&A pipeline."""

    def __init__(self, config: Config):
        self.config = config
        self.ingester = None
        self.chunker = StructureAwareChunker(config)
        self.index = DualIndex(config)
        self.reranker = Reranker(config)
        self.generator = AnswerGenerator(config)
        self.chunks = []

    def ingest(self, pdf_path: str):
        """Ingest a PDF document and build indices."""
        print(f"\n{'='*60}")
        print(f"INGESTING: {pdf_path}")
        print(f"{'='*60}")

        # Step 1: Parse PDF
        self.ingester = DocumentIngester(pdf_path)
        elements = self.ingester.ingest()

        # Step 2: Chunk
        self.chunks = self.chunker.chunk(elements)

        # Step 3: Build index
        self.index.build(self.chunks)

        print(f"\n{'='*60}")
        print(f"✅ Pipeline ready. {len(self.chunks)} chunks indexed.")
        print(f"{'='*60}")

    def ask(self, question: str, verbose: bool = True) -> Dict[str, Any]:
        """
        Ask a question about the ingested document.

        Args:
            question: The user's question
            verbose: Whether to print detailed output

        Returns:
            Result dict with answer, sources, and metadata
        """
        if not self.chunks:
            return {"answer": "No document ingested. Call pipeline.ingest() first."}

        if verbose:
            print(f"\n{'─'*60}")
            print(f"Q: {question}")
            print(f"{'─'*60}")

        # Step 1: Hybrid retrieval
        candidates = self.index.search_hybrid(question, self.config.top_k_initial)
        if verbose:
            print(f"\n📚 Retrieved {len(candidates)} candidates")

        # Step 2: Re-rank
        if self.config.use_reranker:
            passages = self.reranker.rerank(question, candidates, self.config.top_k_rerank)
            if verbose:
                print(f"🔄 Re-ranked to top {len(passages)}")
        else:
            passages = candidates[:self.config.top_k_rerank]

        # Step 3: Generate answer
        if verbose:
            print(f"🤖 Generating answer with {self.config.llm_model}...")

        result = self.generator.generate(question, passages)

        if verbose:
            print(f"\n💡 ANSWER:\n{result['answer']}")
            print(f"\n📖 Sources used:")
            for s in result["sources"]:
                chunk = s["chunk"]
                print(f"   [Source {s['source_num']}] {chunk.citation_str()}")
                print(f"      {chunk.text[:80].replace(chr(10), ' ')}...")

        return result

    def ask_batch(self, questions: List[str]) -> List[Dict[str, Any]]:
        """Ask multiple questions and collect results."""
        results = []
        for i, q in enumerate(questions, 1):
            print(f"\n[{i}/{len(questions)}]")
            result = self.ask(q, verbose=True)
            results.append(result)
        return results

print("DocumentQAPipeline defined.")

DocumentQAPipeline defined.


In [27]:
# ── Initialize and Run Pipeline ───────────────────────────────

PDF_PATH = "document.pdf"  # ← CHANGE THIS

pipeline = DocumentQAPipeline(config)

if os.path.exists(PDF_PATH):
    pipeline.ingest(PDF_PATH)
else:
    print(f"⚠️  Set PDF_PATH to your document. Current: {PDF_PATH}")

Loading re-ranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 17220.02it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Re-ranker loaded.

INGESTING: document.pdf
✅ Ingested 160 pages → 816 elements
   heading: 185
   paragraph: 626
   table: 5
✅ Created 348 chunks
   Token stats: min=52, max=538, mean=398, median=476
Building dense index...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12334.55it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 11/11 [00:03<00:00,  2.83it/s]

   FAISS index: 348 vectors, 768 dimensions
Building sparse index...
   BM25 index: 348 documents
✅ Dual index built: 348 chunks indexed

✅ Pipeline ready. 348 chunks indexed.


In [28]:
# ── Ask Questions ─────────────────────────────────────────────
# Replace these with questions relevant to your document.

if pipeline.chunks:
    # Example questions — adapt to your document
    questions = [
        "What was the total revenue for the fiscal year?",
        "What are the main risk factors mentioned in the document?",
        "Who are the key executives listed in the document?",
    ]

    for q in questions:
        result = pipeline.ask(q)
        print()


────────────────────────────────────────────────────────────
Q: What was the total revenue for the fiscal year?
────────────────────────────────────────────────────────────

📚 Retrieved 20 candidates
🔄 Re-ranked to top 8
🤖 Generating answer with gpt-4o-mini...

💡 ANSWER:
The total revenue for the fiscal year 2018 was $32,765 million, as stated in the sales results by geographic area/business segment [Source 8].

📖 Sources used:
   [Source 1] [p. 121 — "Net income attributable to 3M
 
$
5,349  
$
4,858  
$
5,050"]
       of income. These amounts totaled $100 million, $228 million, and $184 million f...
   [Source 2] [p. 43 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"]
       currency effects. These were partially offset by August 2018 and November 2018 ...
   [Source 3] [p. 72 — "Net income attributable to 3M
 
$
5,349  
$
4,858  
$
5,050"]
      relating to the accelera

---
## Section 8: Evaluation Framework

### Evaluation Approach
Since we don't have gold-standard labels, we use a multi-faceted evaluation:

1. **Retrieval Quality** — Are the right passages being retrieved?
   - Manual inspection of top-k results for sample queries
   - Compare dense-only vs sparse-only vs hybrid

2. **Answer Quality** — Is the answer correct, grounded, and well-cited?
   - **Faithfulness**: Does the answer only state things in the sources?
   - **Relevance**: Does the answer address the question?
   - **Citation accuracy**: Do citations point to passages that support the claims?

3. **Robustness Tests**
   - Questions with no answer in the document (should abstain)
   - Questions requiring multi-hop reasoning across sections
   - Questions about specific numbers/tables

### LLM-as-Judge
We use a separate LLM call to evaluate answer quality on three dimensions.
This is not a substitute for human evaluation but provides a scalable signal.

In [29]:
class Evaluator:
    """Evaluate the QA pipeline on multiple dimensions."""

    EVAL_PROMPT = """You are evaluating a document Q&A system. Given the question, the system's
answer, and the source passages that were retrieved, rate the answer on three dimensions.

QUESTION: {question}

RETRIEVED SOURCES:
{sources}

SYSTEM ANSWER:
{answer}

Rate each dimension from 1 (worst) to 5 (best):

1. FAITHFULNESS: Does the answer only contain information supported by the sources?
   5 = Fully grounded, every claim traceable to sources
   3 = Mostly grounded but some minor unsupported claims
   1 = Contains significant hallucination or unsupported claims

2. RELEVANCE: Does the answer address the question asked?
   5 = Directly and completely answers the question
   3 = Partially answers, misses some aspects
   1 = Does not address the question

3. CITATION_QUALITY: Are citations present, correct, and sufficient?
   5 = All claims cited, citations match source content
   3 = Some citations present but incomplete
   1 = No citations or citations don't match

Respond in this exact JSON format:
{{"faithfulness": <1-5>, "relevance": <1-5>, "citation_quality": <1-5>, "reasoning": "<brief explanation>"}}"""

    def __init__(self, config: Config):
        self.config = config
        self.client = None
        if config.openai_api_key:
            self.client = OpenAI(api_key=config.openai_api_key)

    def evaluate_single(self, result: Dict[str, Any]) -> Dict[str, Any]:
        """Evaluate a single QA result using LLM-as-judge."""
        if not self.client:
            return {"error": "No API key configured"}

        # Format sources for the evaluator
        sources_text = ""
        for s in result["sources"]:
            chunk = s["chunk"]
            sources_text += f"[Source {s['source_num']}] {chunk.citation_str()}\n"
            sources_text += f"{chunk.text[:500]}\n\n"

        prompt = self.EVAL_PROMPT.format(
            question=result["query"],
            sources=sources_text,
            answer=result["answer"],
        )

        response = self.client.chat.completions.create(
            model=self.config.llm_model,
            temperature=0.0,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
        )

        try:
            eval_text = response.choices[0].message.content
            # Extract JSON from response
            json_match = re.search(r'\{[^}]+\}', eval_text, re.DOTALL)
            if json_match:
                scores = json.loads(json_match.group())
                return scores
        except (json.JSONDecodeError, AttributeError):
            pass

        return {"error": "Failed to parse evaluation response", "raw": eval_text}

    def evaluate_batch(self, results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Evaluate a batch of QA results and compute aggregate metrics."""
        evaluations = []
        for i, result in enumerate(results):
            print(f"  Evaluating {i+1}/{len(results)}: {result['query'][:50]}...")
            eval_result = self.evaluate_single(result)
            eval_result["query"] = result["query"]
            evaluations.append(eval_result)

        # Compute aggregates
        valid_evals = [e for e in evaluations if "error" not in e]
        if valid_evals:
            summary = {
                "num_evaluated": len(valid_evals),
                "avg_faithfulness": np.mean([e["faithfulness"] for e in valid_evals]),
                "avg_relevance": np.mean([e["relevance"] for e in valid_evals]),
                "avg_citation_quality": np.mean([e["citation_quality"] for e in valid_evals]),
                "details": evaluations,
            }
        else:
            summary = {"num_evaluated": 0, "details": evaluations}

        return summary

    def retrieval_comparison(
        self, query: str, index: DualIndex, top_k: int = 5
    ) -> Dict[str, List]:
        """Compare dense-only, sparse-only, and hybrid retrieval."""
        dense = index.search_dense(query, top_k)
        sparse = index.search_sparse(query, top_k)
        hybrid = index.search_hybrid(query, top_k)

        return {
            "query": query,
            "dense": [(index.chunks[idx].citation_str(), index.chunks[idx].text[:100])
                      for idx, _ in dense],
            "sparse": [(index.chunks[idx].citation_str(), index.chunks[idx].text[:100])
                       for idx, _ in sparse],
            "hybrid": [(c.citation_str(), c.text[:100]) for c, _ in hybrid],
        }

print("Evaluator defined.")

Evaluator defined.


In [30]:
# ── Run Evaluation ────────────────────────────────────────────

if pipeline.chunks and config.openai_api_key:
    evaluator = Evaluator(config)

    # Define test questions — CUSTOMIZE these for your document
    eval_questions = [
        # Factual questions (should have clear answers)
        "What was the total revenue for the most recent fiscal year?",
        "What are the main business segments described in the document?",

        # Table/numerical questions
        "What were the operating expenses?",

        # Multi-section questions
        "What are the key risk factors and how do they relate to the company's strategy?",

        # Unanswerable question (should trigger abstention)
        "What is the company's plan for colonizing Mars?",
    ]

    print("Running evaluation pipeline...\n")
    results = pipeline.ask_batch(eval_questions)

    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print("="*60)

    summary = evaluator.evaluate_batch(results)

    if summary["num_evaluated"] > 0:
        print(f"\n📊 Aggregate Scores ({summary['num_evaluated']} questions):")
        print(f"   Faithfulness    : {summary['avg_faithfulness']:.1f}/5")
        print(f"   Relevance       : {summary['avg_relevance']:.1f}/5")
        print(f"   Citation Quality: {summary['avg_citation_quality']:.1f}/5")

        print(f"\n📋 Per-question breakdown:")
        for e in summary["details"]:
            q = e.get("query", "?")[:50]
            if "error" not in e:
                print(f"   {q}...")
                print(f"     Faith={e['faithfulness']} Rel={e['relevance']} Cite={e['citation_quality']}")
                print(f"     {e.get('reasoning', '')[:80]}")
            else:
                print(f"   {q}... → Error: {e['error']}")
else:
    if not pipeline.chunks:
        print("⚠️  Ingest a document first.")
    if not config.openai_api_key:
        print("⚠️  Set OPENAI_API_KEY for evaluation.")

Running evaluation pipeline...


[1/5]

────────────────────────────────────────────────────────────
Q: What was the total revenue for the most recent fiscal year?
────────────────────────────────────────────────────────────

📚 Retrieved 20 candidates
🔄 Re-ranked to top 8
🤖 Generating answer with gpt-4o-mini...

💡 ANSWER:
The total revenue for the most recent fiscal year, which is 2018, was $32,765 million [Source 2, Source 4].

📖 Sources used:
   [Source 1] [p. 72 — "Net income attributable to 3M
 
$
5,349  
$
4,858  
$
5,050"]
      relating to the accelerated recognition for software installation service and tr...
   [Source 2] [p. 22 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"]
      were 21.3 percent, down 1.1 percentage points, which included an increase of 0.5...
   [Source 3] [p. 25 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification 

In [ ]:
# ── Retrieval Comparison ──────────────────────────────────────
# Compare how different retrieval methods perform on the same query.

if 'dual_index' in dir() and pipeline.chunks:
    evaluator = Evaluator(config)
    test_q = "What was the total revenue?"  # ← Adjust for your document

    comparison = evaluator.retrieval_comparison(test_q, dual_index, top_k=5)

    print(f"Query: '{comparison['query']}'\n")
    for method in ["dense", "sparse", "hybrid"]:
        print(f"--- {method.upper()} ---")
        for citation, preview in comparison[method][:3]:
            print(f"  {citation}: {preview.replace(chr(10), ' ')}...")
        print()

---
## Section 9: Interactive Q&A

Run the cell below to start an interactive question-answering session.
Type your questions and get grounded answers with citations.
Type `quit` to exit.

In [31]:
# ── Interactive Q&A Loop ──────────────────────────────────────
def interactive_qa(pipeline):
    """Interactive Q&A loop."""
    print("╔══════════════════════════════════════════════════════════╗")
    print("║        Document Q&A System — Interactive Mode              ║")
    print("║  Type your question and press Enter.                    ║")
    print("║  Type 'quit' to exit.                                   ║")
    print("╚══════════════════════════════════════════════════════════╝")

    while True:
        print()
        question = input("❓ Your question: ").strip()
        if question.lower() in ("quit", "exit", "q"):
            print("👋 Goodbye!")
            break
        if not question:
            continue

        result = pipeline.ask(question, verbose=True)

# Uncomment the line below to start the interactive session:
# interactive_qa(pipeline)

---
## Section 10: Design Decisions, Limitations & Future Work

### Architecture Summary

```
PDF Document
    │
    ▼
┌─────────────────┐
│  PDF Parser     │  PyMuPDF: text, headings, tables
│  (Ingestion)    │  with page-level metadata
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Structure-Aware│  Heading-first grouping
│  Chunker        │  Token-bounded with overlap
└────────┬────────┘
         │
         ▼
┌─────────────────────────────────┐
│  Dual Index                     │
│  ┌──────────┐  ┌──────────────┐ │
│  │  FAISS   │  │  BM25        │ │
│  │  (dense) │  │  (sparse)    │ │
│  └──────────┘  └──────────────┘ │
└────────┬────────────────────────┘
         │
         ▼
┌─────────────────┐
│  RRF Fusion     │  Reciprocal Rank Fusion
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Cross-Encoder  │  Re-rank for precision
│  Re-ranker      │  (optional)
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  LLM Generator  │  Strict grounding prompt
│  + Citations    │  with source attribution
└─────────────────┘
```

### Key Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| PDF Parser | PyMuPDF | Speed, structure extraction, table support |
| Chunking | Heading-aware + token overlap | Preserves topical coherence |
| Embeddings | bge-base-en-v1.5 | Strong retrieval quality, manageable size |
| Vector Store | FAISS (flat) | Single document = small corpus, exact search is fine |
| Sparse Retrieval | BM25 | Essential for exact term matching |
| Fusion | RRF (k=60) | Score-agnostic, no normalization needed |
| Re-ranking | cross-encoder/ms-marco-MiniLM | Good accuracy/latency tradeoff |
| LLM | gpt-4o-mini | Cost-effective, strong instruction following |
| Citations | Source numbering + page refs | Clear, verifiable grounding |

### Limitations

1. **Table extraction**: PyMuPDF tables work for simple layouts; complex merged cells may fail. For production, consider Camelot or Tabula as fallbacks.
2. **Image/chart content**: We extract text only. Figures, charts, and images are ignored. A multimodal approach (GPT-4V, etc.) could handle these.
3. **Cross-page reasoning**: Chunks may miss information that spans multiple distant sections. Larger context windows or iterative retrieval could help.
4. **Heading detection**: Font-size heuristics work for most documents but may fail on unusual layouts. A more robust approach would use the PDF's outline/bookmark structure.
5. **Evaluation**: LLM-as-judge has known biases. Human evaluation would be more reliable.
6. **Single document**: Designed for one document at a time. Multi-document support would need namespace management and potentially different fusion strategies.

### Domain Generalization

While tested on finance documents, the system generalizes because:
- No domain-specific rules or ontologies are used
- Chunking is structure-based, not content-based
- Embeddings are general-purpose
- The generation prompt is domain-agnostic

To specialize for a domain, you could:
- Fine-tune embeddings on domain data
- Add domain-specific preprocessing (e.g., chemical formulas, legal citations)
- Customize the generation prompt with domain constraints

### Future Improvements (with more time)

1. **Iterative retrieval**: If the first retrieval doesn't find good evidence, reformulate the query and search again.
2. **Semantic chunking**: Use embedding similarity between sentences to find natural breakpoints.
3. **Agentic QA**: Let the LLM decide when it needs more context and trigger additional retrieval.
4. **Caching**: Serialize the FAISS index and embeddings to disk so re-ingestion isn't needed.
5. **Streaming**: Stream the LLM response for better UX.
6. **Multi-modal**: Process charts and images using vision models.
7. **Confidence scores**: Estimate answer reliability based on retrieval scores and source coverage.
8. **Query decomposition**: Break complex questions into sub-questions, answer each, then synthesize.